# Imports

In [1]:
!pip install -q pandas openpyxl

In [2]:
import pandas as pd
import numpy as np
import re
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Importing scholarships sheet

In [3]:
class DataIngestionAgent:
    def __init__(self, univ_path, scholarship_path):
        self.univ_path = univ_path
        self.scholarship_path = scholarship_path

    def load_universities(self, domain_name):
        print(f"[Ingestion] Accessing sheet: {domain_name}...")
        df = pd.read_excel(self.univ_path, sheet_name=domain_name, header=3)
        df.columns = df.columns.astype(str).str.strip()

        if 'INSTITUTION' in df.columns:
            df.rename(columns={'INSTITUTION': 'University / Partners'}, inplace=True)
        return df

    def load_scholarships(self):
        df = pd.read_excel(self.scholarship_path)
        df.columns = df.columns.astype(str).str.strip()
        return df

# User profile

In [4]:
class ProfilingAgent:
    def run(self, user_data):
        return {
            "domain": str(user_data.get("domain", "")),
            "gpa": float(user_data.get("gpa", 0.0)),
            "ielts": float(user_data.get("ielts", 0.0)),
            "degree": str(user_data.get("degree_level", "Masters")),
            "gre": str(user_data.get("gre", "no")).strip().lower(),
            "experience": str(user_data.get("experience_years", "no")).strip().lower(),
            "willing_to_return": str(user_data.get("willing_to_return", "no")).strip().lower(),
            "graduation_certificate": str(user_data.get("graduation_certificate", "no")).strip().lower()
        }

# Universities Ranking - Top 20

In [5]:
class UniversityFilterAgent:
    def _extract_rank(self, rank_val):
        try:
            match = re.search(r'\d+', str(rank_val))
            return int(match.group()) if match else 99999
        except:
            return 99999

    def run(self, univ_df):
        df = univ_df.copy()
        rank_col = '2026' if '2026' in df.columns else 'Rank'
        df['Numeric_Rank'] = df[rank_col].apply(self._extract_rank)

        # Sort by best rank and take top 20 globally
        top_20 = df.sort_values(by='Numeric_Rank', ascending=True).head(20)
        return top_20

# Scholarships Filter

In [6]:
class ScholarshipFilterAgent:
    def _extract_score(self, score_str):
        score_str = str(score_str).lower()
        if 'not required' in score_str or 'none' in score_str:
            return 0.0

        matches = re.findall(r'\b([4-9](?:\.\d)?)\b', score_str)
        if matches:
            return float(matches[0])
        return 0.0

    def run(self, scholarship_df, profile):
        df = scholarship_df.copy()

        # 1. Degree Level
        df = df[df['Degree Level'].str.contains(profile['degree'], case=False, na=False)]

        # 2. IELTS Requirement (Flexible: <= user's score)
        df['Parsed_IELTS'] = df['Min IELTS / TOEFL'].apply(self._extract_score)
        df = df[df['Parsed_IELTS'] <= profile['ielts']]

        # 3. Field Restrictions
        def field_match(x):
            x_str = str(x).lower()
            if 'open' in x_str or 'none' in x_str:
                return True
            domain_words = profile['domain'].lower().replace('&', '').split()
            return any(len(w) > 3 and w in x_str for w in domain_words)

        df = df[df['Field Restrictions'].apply(field_match)]

        # --- FLEXIBLE ELIGIBILITY FILTERS ---

        # 4. GRE Filter
        # If YES: they see everything. If NO: they only see "Not Required"
        if profile['gre'] in ["no", "false"]:
            df = df[df['GRE Required?'].str.contains("Not Required", case=False, na=False)]

        # 5. Experience Requirement Filter
        # If YES: they see both required and not required. If NO: drop the ones requiring it.
        if profile['experience'] in ["no", "0", "false"]:
            df = df[~df['Experience Required'].str.contains('Yes', case=False, na=False)]

        # 6. Return Obligation Filter
        # If YES (willing): they see both. If NO (unwilling): drop the ones requiring return.
        if profile['willing_to_return'] in ["no", "false"]:
            df = df[~df['Return Obligation'].str.contains('Yes', case=False, na=False)]

        # 7. Graduation Certificate Filter
        # If YES (has it): they see both. If NO (doesn't have it): drop the ones requiring it at application.
        if profile['graduation_certificate'] in ["no", "false"]:
            df = df[~df['Grad Certificate Required at Application?'].str.contains('Yes', case=False, na=False)]

        return df

# Does the scholarship have a top 20 university in the domain?


In [7]:
class MatchingAgent:
    def __init__(self):
        self.country_mapping = {
            'United Kingdom': ['uk', 'united kingdom', 'britain', 'british'],
            'United States of America': ['us', 'usa', 'united states', 'american'],
            'France': ['france', 'french'],
            'Germany': ['germany', 'german'],
            'Italy': ['italy', 'italian'],
            'China (Mainland)': ['china', 'chinese'],
            'Japan': ['japan', 'japanese'],
            'South Korea': ['korea', 'korean'],
            'Australia': ['australia', 'australian'],
            'Canada': ['canada', 'canadian'],
            'Türkiye': ['turkey', 'turkish', 'türkiye'],
            'Russian Federation': ['russia', 'russian'],
            'Egypt': ['egypt', 'egyptian'],
            'Netherlands': ['netherlands', 'dutch'],
            'Switzerland': ['switzerland', 'swiss'],
            'Singapore': ['singapore']
        }

    def run(self, filtered_univs, filtered_scholarships, profile):
        univ_ranks = {}
        country_ranks = {}

        # Store University and Country QS Ranks
        for idx, row in filtered_univs.iterrows():
            u_name = str(row['University / Partners']).lower().strip()
            rank = row['Numeric_Rank']
            univ_ranks[u_name] = rank

            if 'COUNTRY/TERRITORY' in row:
                c_name = str(row['COUNTRY/TERRITORY']).lower().strip()
                if c_name not in country_ranks or rank < country_ranks[c_name]:
                    country_ranks[c_name] = rank

        # Expand country keywords (e.g. "British", "UK" maps to United Kingdom's rank)
        expanded_country_ranks = {}
        for c_name, rank in country_ranks.items():
            expanded_country_ranks[c_name] = rank
            for key_country, keywords in self.country_mapping.items():
                if key_country.lower() in c_name or c_name in key_country.lower():
                    for kw in keywords:
                        expanded_country_ranks[kw] = rank

        def get_best_rank(sch_univ):
            sch_str = str(sch_univ).lower()
            best_rank = 99999

            # Exact University Match
            for u_name, rank in univ_ranks.items():
                if u_name in sch_str or sch_str in u_name:
                    if rank < best_rank:
                        best_rank = rank

            # Country Match (only valid if country has a top 20 univ)
            if any(w in sch_str for w in ['all ', 'most ', 'designated', 'universities in', 'institutions']):
                for kw, rank in expanded_country_ranks.items():
                    if re.search(r'\b' + re.escape(kw) + r'\b', sch_str):
                        if rank < best_rank:
                            best_rank = rank

            # Always allow explicit local fellowships for Egypt
            if 'egypt' in sch_str and best_rank == 99999:
                return 99990

            return best_rank
        df = filtered_scholarships.copy()

        # Link every scholarship to a QS Rank
        df['Matched_Rank'] = df['University / Partners'].apply(get_best_rank)

        # EXPLICITLY DELETE anything that did not match a top 20 university/country
        valid_matches = df[df['Matched_Rank'] < 99999].copy()

        # Sort from Best Rank (1) to Worst Rank
        valid_matches = valid_matches.sort_values(by='Matched_Rank', ascending=True)

        # Return only up to 5 best options
        return valid_matches.head(5).drop(columns=['Matched_Rank', 'Parsed_IELTS'], errors='ignore')


# Orchestrator

In [8]:
class EgyptianScholarshipSystem:
    def __init__(self, univ_file, scholarship_file):
        self.ingestor = DataIngestionAgent(univ_file, scholarship_file)
        self.profiler = ProfilingAgent()
        self.univ_filter = UniversityFilterAgent()
        self.sch_filter = ScholarshipFilterAgent()
        self.matcher = MatchingAgent()

        self.scholarship_data = self.ingestor.load_scholarships()

    def process_request(self, user_input):
        profile = self.profiler.run(user_input)
        univ_data = self.ingestor.load_universities(profile['domain'])

        top_univs = self.univ_filter.run(univ_data)
        possible_scholarships = self.sch_filter.run(self.scholarship_data, profile)

        final_results = self.matcher.run(top_univs, possible_scholarships, profile)
        return final_results

# Main cell

In [9]:
system = EgyptianScholarshipSystem("/content/Universities.xlsx", "/content/Scholarships.xlsx")

user_query = {
    "domain": "Data Science and Artificial Int",
    "gpa": 3.9,
    "ielts": 6,
    "degree_level": "Masters",
    "gre": "no",
    "experience_years": "no",
    "willing_to_return": "no",
    "graduation_certificate" :"yes"
}

top_5 = system.process_request(user_query)

print("\n=== TOP 5 PERSONALIZED SCHOLARSHIPS ===")
display(top_5[[
    'Scholarship Name',
    'University / Partners',
    'Funding Type',
    'Deadline Month'
]])

[Ingestion] Accessing sheet: Data Science and Artificial Int...

=== TOP 5 PERSONALIZED SCHOLARSHIPS ===


,Scholarship Name,University / Partners,Funding Type,Deadline Month
5,CSC – Chinese Government Scholarship,Chinese Universities (designated list),Full,April
